<a href="https://colab.research.google.com/github/hiapaul/BankingSystemC/blob/main/Braintumor_using_VGG16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import all libraries

In [47]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam


Image processing :

In [48]:
IMG_SIZE = 128
BATCH_SIZE = 64
EPOCHS = 10
LR = 0.0001
CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']

print("=" * 55)
print("   OPTIMIZED TRAINING CONFIGURATION")
print("=" * 55)
print(f"  Image Size  : {IMG_SIZE}×{IMG_SIZE} (reduced for speed)")
print(f"  Batch Size  : {BATCH_SIZE}")
print(f"  Epochs      : {EPOCHS}")
print(f"  Total Steps : ~{BATCH_SIZE * EPOCHS} per epoch")


   OPTIMIZED TRAINING CONFIGURATION
  Image Size  : 128×128 (reduced for speed)
  Batch Size  : 64
  Epochs      : 10
  Total Steps : ~640 per epoch


In [49]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

Load dataset

In [50]:
DATASET_PATH = '/content/drive/MyDrive/MRI Image'
TRAIN_DIR = os.path.join('/content/drive/MyDrive/MRI Image/Training', 'Training')
TEST_DIR = os.path.join('/content/drive/MyDrive/MRI Image/Testing', 'Testing')

Build & Train the model

In [ ]:
DATASET_PATH = '/content/drive/MyDrive/MRI Image'
TRAIN_DIR = os.path.join(DATASET_PATH, 'Training')
TEST_DIR = os.path.join(DATASET_PATH, 'Testing')

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=42
)

validation_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=42
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


def build_model():
    base_model = tf.keras.applications.VGG16(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )

    for layer in base_model.layers:
        layer.trainable = False

    x = layers.Flatten()(base_model.output)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(len(CLASS_NAMES), activation='softmax')(x)

    model = models.Model(inputs=base_model.input, outputs=outputs)

    model.compile(
        optimizer=Adam(learning_rate=LR),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_model()
model.summary()



early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001)


try:
    history = model.fit(
        train_generator,
        validation_data=validation_generator,
        epochs=EPOCHS,
        callbacks=[early_stopping, reduce_lr]
    )
except Exception as e:
    print(f"An error occurred during model training: {e}")

Found 4480 images belonging to 4 classes.
Found 1120 images belonging to 4 classes.
Found 1614 images belonging to 4 classes.


Model: "functional_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_9 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 128, 128, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 128, 128, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 64, 64, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 32, 32, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 16, 16, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 16, 16, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 8, 8, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 8, 8, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_6 (Flatten)             │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,763,908 (60.13 MB)

 Trainable params: 1,049,220 (4.00 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

Epoch 1/10
15/70 ━━━━━━━━━━━━━━━━━━━━ 11:07 12s/step - accuracy: 0.4111 - loss: 1.3256

Evalute the model

In [ ]:
print("\nEvaluating model on the test set...")
test_loss, test_accuracy = model.evaluate(test_generator)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Confusion matrix & classification report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import seaborn as sns
import matplotlib.pyplot as plt


test_generator.reset()
y_true = []
y_pred = []
y_pred_probs = []

for _ in range(len(test_generator)):
    X, y = next(test_generator)
    y_true.extend(np.argmax(y, axis=1))
    batch_predictions = model.predict(X, verbose=0)
    y_pred.extend(np.argmax(batch_predictions, axis=1))
    y_pred_probs.extend(batch_predictions)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_pred_probs = np.array(y_pred_probs)


print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))


conf_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

ROC Curves

In [ ]:
def plot_roc_curves(y_true, y_pred_probs, class_names):
    y_bin  = label_binarize(y_true, classes=list(range(len(class_names))))
    colors = ['#e74c3c', '#e67e22', '#27ae60', '#2980b9']
    plt.figure(figsize=(9, 7))
    for i, (cls, col) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_probs[:, i])
        plt.plot(fpr, tpr, color=col, lw=2,
                 label=f'{cls.upper()} (AUC = {auc(fpr,tpr):.3f})')
    plt.plot([0,1],[0,1],'k--',lw=1.5, label='Random')
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate',  fontsize=12)
    plt.title('ROC Curves — CNN (Optimized)', fontsize=14, fontweight='bold')
    plt.legend(loc='lower right', fontsize=11); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('/content/roc_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_roc_curves(y_true, y_pred_probs, CLASS_NAMES)

Model Prediction Visualization (True vs Predicted Labels)

In [ ]:
import matplotlib.pyplot as plt

test_generator.reset()
x_test, y_test = next(test_generator)


predictions = model.predict(x_test)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(y_test, axis=1)

num_images_to_show = 9
plt.figure(figsize=(10, 10))
for i in range(num_images_to_show):
    plt.subplot(3, 3, i + 1)
    plt.imshow(x_test[i])
    plt.title(f"True: {CLASS_NAMES[true_classes[i]]}\nPred: {CLASS_NAMES[predicted_classes[i]]}",
              color='green' if true_classes[i] == predicted_classes[i] else 'red')
    plt.axis('off')
plt.tight_layout()
plt.show()

Single Image Prediction with Confidence Scores

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

def predict_single_image(img_path, model, class_names):
    img     = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_arr = img_to_array(img) / 255.0
    inp     = np.expand_dims(img_arr, axis=0)
    preds   = model.predict(inp, verbose=0)
    pred_cls = class_names[np.argmax(preds)]
    conf    = np.max(preds) * 100
    print(f"\n  Prediction  : {pred_cls.upper()}  ({conf:.2f}% confidence)")
    for c, s in zip(class_names, preds[0]):
        print(f"  {c:<15} {'█' * int(s*30)} {s:.4f}")
    return pred_cls, conf

img_path = '/content/drive/MyDrive/MRI Image/Testing/glioma/Te-gl_101.jpg'

predicted_class, confidence = predict_single_image(img_path, model, CLASS_NAMES)

Bar chart per class

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


report_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)

metrics_data = []
for class_name in CLASS_NAMES:
    metrics_data.append({
        'Class': class_name,
        'Metric': 'Precision',
        'Score': report_dict[class_name]['precision']
    })
    metrics_data.append({
        'Class': class_name,
        'Metric': 'Recall',
        'Score': report_dict[class_name]['recall']
    })
    metrics_data.append({
        'Class': class_name,
        'Metric': 'F1-score',
        'Score': report_dict[class_name]['f1-score']
    })
metrics_df = pd.DataFrame(metrics_data)

plt.figure(figsize=(12, 7))
sns.barplot(x='Class', y='Score', hue='Metric', data=metrics_df, palette='viridis')
plt.title('Classification Report Metrics Per Class')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.legend(title='Metric')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

Model accuracy and loss

In [ ]:
import matplotlib.pyplot as plt


plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.tight_layout()
plt.show()

 Visualization of Training History and ROC Curves

In [ ]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import numpy as np
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.tight_layout()
plt.show()

def plot_roc_curves(y_true, y_pred_probs, class_names):
    y_bin  = label_binarize(y_true, classes=list(range(len(class_names))))
    colors = ['#e74c3c', '#e67e22', '#27ae60', '#2980b9']
    plt.figure(figsize=(9, 7))
    for i, (cls, col) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_probs[:, i])
        plt.plot(fpr, tpr, color=col, lw=2,
                 label=f'{cls.upper()} (AUC = {auc(fpr,tpr):.3f})')
    plt.plot([0,1],[0,1],'k--',lw=1.5, label='Random')
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate',  fontsize=12)
    plt.title('ROC Curves — CNN (Optimized)', fontsize=14, fontweight='bold')
    plt.legend(loc='lower right', fontsize=11); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_roc_curves(y_true, y_pred_probs, CLASS_NAMES)

 Single Image Prediction (another one)


In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np

def predict_single_image(img_path, model, class_names):
    img     = load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_arr = img_to_array(img) / 255.0
    inp     = np.expand_dims(img_arr, axis=0)
    preds   = model.predict(inp, verbose=0)
    pred_cls = class_names[np.argmax(preds)]
    conf    = np.max(preds) * 100
    print(f"\n  Prediction  : {pred_cls.upper()}  ({conf:.2f}% confidence)")
    for c, s in zip(class_names, preds[0]):
        print(f"  {c:<15} {'█' * int(s*30)} {s:.4f}")
    return pred_cls, conf

img_path = '/content/drive/MyDrive/MRI Image/Testing/glioma/Te-gl_101.jpg'
predicted_class, confidence = predict_single_image(img_path, model, CLASS_NAMES)

Final

In [ ]:
import time
secs = 0
print("\n" + "=" * 55)
print("  FINAL SUMMARY — CNN OPTIMIZED")
print("=" * 55)
print(f"  Test Accuracy  : {test_accuracy * 100:.2f}%")
print(f"  Test Loss      : {test_loss:.4f}")
print(f"  Training Time  : {mins}m {secs}s")
print(f"  Parameters     : {model.count_params():,}")
print(f"  Epochs run     : {len(history.history['accuracy'])}")
print()
print("  Saved outputs:")
print("  ✔ /content/roc_curves.png")
print("  ✔ Per class metrics plot (generated but not explicitly saved to file)")
print("  ✔ Confusion matrix plot (generated but not explicitly saved to file)")
print("  ✔ Model accuracy and loss plots (generated but not explicitly saved to file)")
print("=" * 55)